Importing packages

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib


In [4]:
df = pd.read_csv('crop_yield_data.csv')

print(df.head())

   rainfall_mm  soil_quality_index  farm_size_hectares  sunlight_hours  \
0         1626                   9                 636              11   
1         1959                   9                  73              11   
2         1360                   1                 352               5   
3         1794                   2                 948               7   
4         1630                   5                 884               5   

   fertilizer_kg  crop_yield  
0           1006         404  
1            112         115  
2            702         231  
3            299         537  
4           2733         554  


In [5]:
X = df.drop('crop_yield', axis=1)
y = df['crop_yield']


In [6]:


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Standardizing X
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [7]:
# Creating model
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1))  # 1 output for regression (yield)

# Compiling model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])


C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 6. Train the model
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 125821.6719 - mae: 325.0305 - val_loss: 121406.0625 - val_mae: 315.7102
Epoch 2/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 115279.3438 - mae: 309.3843 - val_loss: 96438.0234 - val_mae: 279.0125
Epoch 3/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 86686.0859 - mae: 264.5327 - val_loss: 46343.6914 - val_mae: 190.0132
Epoch 4/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 35326.3086 - mae: 163.8740 - val_loss: 7255.5073 - val_mae: 74.7482
Epoch 5/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5837.8203 - mae: 62.5825 - val_loss: 1496.0544 - val_mae: 30.3571
Epoch 6/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3248.3589 - mae: 45.9631 - val_loss: 1377.3842 - val_mae: 29.0394
Epoch 7/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3237.5537 - mae: 45.9109 - val_loss: 1319.6293 - val_mae: 28.3603
Epoch 8/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3136.2886 - mae: 44.4229 - val_loss: 1238.4740 - 

In [9]:
import numpy as np

# Example samples (replace with actual feature names if you want)
sample_inputs = np.array([
    [30.5, 800, 85, 18.0, 200],  # temperature, rainfall, humidity, soil_ph, area
    [25.0, 600, 70, 6.5, 150],
    [35.0, 900, 90, 5.8, 300],
    [28.0, 750, 80, 6.2, 220],
    [22.5, 400, 60, 7.0, 100]
])

# Scale the samples same as training
sample_inputs_scaled = scaler.transform(sample_inputs)

# Predict
predictions = model.predict(sample_inputs_scaled)

for i, pred in enumerate(predictions, 1):
    print(f"Sample {i}: Predicted Crop Yield ➔ {pred[0]:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Sample 1: Predicted Crop Yield ➔ 7316.42
Sample 2: Predicted Crop Yield ➔ 5478.21
Sample 3: Predicted Crop Yield ➔ 8243.65
Sample 4: Predicted Crop Yield ➔ 6860.78
Sample 5: Predicted Crop Yield ➔ 3636.27


C:\Users\Manish Kumar Jain\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [10]:
# Save the trained model to a file
model.save("crop_yield_predictor.keras")
print("✅ Model saved as 'crop_yield_predictor.h5'")

joblib.dump(scaler, "scaler_crop_yield.pkl")
print("✅ Scaler saved as 'scaler_crop_yield.pkl'")

✅ Model saved as 'crop_yield_predictor.h5'
✅ Scaler saved as 'scaler_crop_yield.pkl'
